# prediction_markets API: try it yourself

Every cell talks to the server through `pmapi.py`, which hands back pandas DataFrames.

**Before you start**

1. Put `pmapi.py` in the same folder as this notebook.
2. `pip install pandas pyarrow requests matplotlib` (pyarrow makes downloads ~10x smaller; optional).
3. Your API key in the file `~/.pm_api_key` (one line), or in the environment variable `PM_API_KEY`.
   The notebook reads it from there and never prints it.

A Polymarket market is its condition id (`0x...`) wherever a Kalshi ticker goes.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from pmapi import Client

URL = os.environ.get("PM_API_URL", "https://theonealpha.tail947e08.ts.net:8443")
key_file = Path.home() / ".pm_api_key"
KEY = os.environ.get("PM_API_KEY") or (key_file.read_text().strip() if key_file.exists() else None)
assert KEY, "put your key in ~/.pm_api_key or set PM_API_KEY"

pm = Client(URL, KEY)
pm.health()                      # database, TimescaleDB and Redis up?

## Is collection keeping up?

In [ ]:
s = pm.status()
{k: s[k] for k in s if k in ("kalshi", "poly", "live", "budget")}

## Find markets on both venues

In [ ]:
found = pm.markets(q="bitcoin", status="open", venue="all", limit=20)
found[["venue", "ticker", "title", "close_time", "volume"]]

## One market in full, with its structured record (what the similar search reads)

In [ ]:
k = pm.markets(venue="kalshi", status="open", sort="-volume", limit=1)["ticker"][0]
m = pm.market(k)
{key: m[key] for key in ("ticker", "title", "status", "close_time", "volume", "candles", "proposition")}

## Kalshi candles: any interval, forward-filled where nothing traded

In [ ]:
c = pm.candles(k, "1h")
print(c.attrs.get("coverage"))   # what is stored, and whether it covers the window
c.tail()

In [ ]:
c["close"].plot(title=k, figsize=(10, 3));

## Polymarket: candles made from every trade, and the trades themselves

In [ ]:
p = pm.markets(venue="poly", status="open", sort="-volume", limit=1)["ticker"][0]
print(pm.market(p)["title"])
pc = pm.candles(p, "1h")
pc.tail()

In [ ]:
t = pm.trades(p, limit=2000)     # oldest first: price, side, size, notional (USDC), wallet
t.tail()

## Similar markets on the other venue

In [ ]:
sim = pm.similar(k, venue="poly", status="open", limit=20)
sim[["ticker", "title", "relation", "text", "days", "level", "distance"]]

## Several markets side by side (one close column each), and their correlation

In [ ]:
several = list(found["ticker"][:4])
closes = pm.closes(several, "1h")
closes.corr()

## The last trade of many markets at once

In [ ]:
pm.latest(several)

## Events and categories

In [ ]:
pm.events(venue="poly", q="election", status="open", limit=10)

In [ ]:
pm.categories().head(20)

## The daily data check (09:00 UTC): ok, warn or fail per check

In [ ]:
pm.checks()[["run_at", "name", "status", "value"]]

## Re-download one market now (needs a full key, not a read-only one)

The server answers at once and works in the background; `wait` follows the job.

In [ ]:
# job = pm.backfill(k)
# pm.wait(job)